# **05 — Save & Export Verification**
## **Ma3 | Final Model Registry Check**

Before wiring models into FastAPI, we verify all three
load correctly and produce sensible predictions together —
exactly as the `/predict/*` endpoints will call them.

This notebook is your **pre-flight check** before going live.

Models expected:
| Model | File | Algorithm |
|---|---|---|
| ETA prediction | `eta_model.joblib` | LSTM (Keras wrapped) |
| Demand forecast | `demand_model.joblib` | XGBoost |
| Driver scoring | `score_model.joblib` | Isolation Forest |

In [3]:
import numpy as np
import joblib
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

MODEL_DIR = "../models"

# Must redefine before loading — joblib needs the class in scope
class KerasETAWrapper:
    def __init__(self, keras_model, scaler, features):
        self.model    = keras_model
        self.scaler   = scaler
        self.features = features

    def predict(self, X):
        scaled  = self.scaler.transform(X)
        lstm_in = scaled.reshape(scaled.shape[0], 1, scaled.shape[1])
        return self.model.predict(lstm_in, verbose=0).flatten()

print("Checking model files...")
print("=" * 45)
expected = [
    "eta_model.joblib",
    "eta_scaler.joblib",
    "demand_model.joblib",
    "demand_label_encoder.joblib",
    "score_model.joblib",
    "score_scaler.joblib",
]
all_good = True
for fname in expected:
    path   = os.path.join(MODEL_DIR, fname)
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024 if exists else 0
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {fname:<35} {size:>8.1f} KB")
    if not exists:
        all_good = False
print("=" * 45)
print("All files present ✓" if all_good else "⚠ Some files missing")

Checking model files...
  ✓  eta_model.joblib                       437.2 KB
  ✓  eta_scaler.joblib                        0.9 KB
  ✓  demand_model.joblib                    252.4 KB
  ✓  demand_label_encoder.joblib              0.5 KB
  ✓  score_model.joblib                    2680.0 KB
  ✓  score_scaler.joblib                      0.7 KB
All files present ✓


## **Load All Models**

In [4]:
eta_model    = joblib.load(f"{MODEL_DIR}/eta_model.joblib")
demand_model = joblib.load(f"{MODEL_DIR}/demand_model.joblib")
demand_le    = joblib.load(f"{MODEL_DIR}/demand_label_encoder.joblib")
score_model  = joblib.load(f"{MODEL_DIR}/score_model.joblib")
score_scaler = joblib.load(f"{MODEL_DIR}/score_scaler.joblib")

print("eta_model    →", type(eta_model).__name__)
print("demand_model →", type(demand_model).__name__)
print("score_model  →", type(score_model).__name__)
print("\nAll models loaded ✓")

I0000 00:00:1779592701.416935   12113 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779592707.309819   12113 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


eta_model    → KerasETAWrapper
demand_model → XGBRegressor
score_model  → IsolationForest

All models loaded ✓


## **End-to-End Simulation**

Simulate a full Ma3 event cycle:

1. **Commuter** texts "WESTLANDS" to Ma3 shortcode
2. SMS handler calls `/predict/eta` → LSTM returns ETA
3. SACCO dashboard calls `/predict/demand` → XGBoost returns load
4. End of shift → `/predict/score` → Isolation Forest scores driver
5. All three results assembled into one response object

In [5]:
def predict_eta(hour, day_of_week, stop_sequence, pax_count, speed_kmh, is_peak):
    # eta_model is a KerasETAWrapper — has .predict(X) method
    # Features must match FEATURES list from notebook 02
    # We pass a zero-padded array — only base features matter for this check
    n_features = eta_model.scaler.n_features_in_
    sample = np.zeros((1, n_features))
    sample[0, 0] = hour
    sample[0, 1] = day_of_week
    sample[0, 2] = stop_sequence
    sample[0, 3] = pax_count
    sample[0, 4] = speed_kmh
    sample[0, 5] = is_peak
    return round(float(eta_model.predict(sample)[0]), 1)

eta = predict_eta(hour=8, day_of_week=0, stop_sequence=2,
                  pax_count=10, speed_kmh=15, is_peak=1)
print(f"ETA prediction:  {eta} minutes ✓")

ETA prediction:  12.5 minutes ✓


In [6]:
def predict_demand(route_name, hour, day_of_week, is_holiday=0):
    route_enc = demand_le.transform([route_name])[0]
    hour_sin  = np.sin(2 * np.pi * hour / 24)
    hour_cos  = np.cos(2 * np.pi * hour / 24)
    peak      = 1 if (6 <= hour <= 9) or (17 <= hour <= 20) else 0
    weekend   = 1 if day_of_week >= 5 else 0
    features  = np.array([[hour, hour_sin, hour_cos,
                           day_of_week, peak, weekend,
                           is_holiday, route_enc]])
    pax   = float(demand_model.predict(features)[0])
    level = "HIGH" if pax > 60 else "MEDIUM" if pax > 30 else "LOW"
    return round(pax, 1), level

pax, level = predict_demand("46 CBD-Westlands", hour=8, day_of_week=0)
print(f"Demand prediction: {pax} pax → {level} ✓")

Demand prediction: 78.8 pax → HIGH ✓


In [7]:
def score_trip(speed_variance, off_route_ratio,
               avg_dwell_time, harsh_braking, duration):
    features = np.array([[speed_variance, off_route_ratio,
                          avg_dwell_time, harsh_braking, duration]])
    scaled = score_scaler.transform(features)
    raw    = -score_model.score_samples(scaled)[0]
    min_s  = score_model._train_score_min
    max_s  = score_model._train_score_max
    score  = float(np.clip((1 - (raw - min_s) / (max_s - min_s)) * 100, 0, 100))
    status = "Safe" if score >= 80 else "Monitor" if score >= 50 else "Flagged"
    return round(score, 1), status

score, status = score_trip(6.1, 0.03, 22.0, 1, 28.0)
print(f"Driver score:      {score}/100 → {status} ✓")

Driver score:      96.2/100 → Safe ✓


## **Full Event Cycle — Assembled Response**

This is exactly what FastAPI returns when all three
endpoints are called for a single vehicle on a single trip.

In [8]:
from datetime import datetime

plate    = "KDA 123A"
phone    = "+254700000001"
route    = "46 CBD-Westlands"
now      = datetime.now()

eta_min          = predict_eta(now.hour, now.weekday(), 2, 10, 15, 1)
demand_pax, load = predict_demand(route, now.hour, now.weekday())
drv_score, flag  = score_trip(6.1, 0.03, 22.0, 1, 28.0)

response = {
    "vehicle":  plate,
    "route":    route,
    "timestamp": now.strftime("%Y-%m-%d %H:%M"),
    "eta": {
        "minutes": eta_min,
        "sms_text": f"[Ma3] {route} inakuja Westlands in ~{eta_min} min. KSh 50. Lipa: *384*1#"
    },
    "demand": {
        "expected_pax": demand_pax,
        "load_level":   load,
        "redeploy_alert": load == "HIGH"
    },
    "driver": {
        "score":  drv_score,
        "status": flag,
        "ussd_summary": f"Leo score yako: {drv_score}/100. {('Hongera!' if drv_score >= 80 else 'Jaribu zaidi.')}"
    }
}

print("=" * 55)
print("MA3 — FULL EVENT CYCLE RESPONSE")
print("=" * 55)
for section, data in response.items():
    if isinstance(data, dict):
        print(f"\n[{section.upper()}]")
        for k, v in data.items():
            print(f"  {k:<20} {v}")
    else:
        print(f"  {section:<20} {data}")
print("=" * 55)

MA3 — FULL EVENT CYCLE RESPONSE
  vehicle              KDA 123A
  route                46 CBD-Westlands
  timestamp            2026-05-24 06:20

[ETA]
  minutes              12.4
  sms_text             [Ma3] 46 CBD-Westlands inakuja Westlands in ~12.4 min. KSh 50. Lipa: *384*1#

[DEMAND]
  expected_pax         46.9
  load_level           MEDIUM
  redeploy_alert       False

[DRIVER]
  score                96.2
  status               Safe
  ussd_summary         Leo score yako: 96.2/100. Hongera!


## **Update FastAPI ml/loader.py**

The `predict.py` router in FastAPI calls `loader.py` to get models.
We need to update it to use the demand label encoder and
score scaler correctly — exactly as tested in this notebook.

In [9]:
loader_code = '''
import joblib, os, numpy as np

MODEL_DIR = os.getenv("MODEL_DIR", "/app/ml_models")

def _load(name):
    path = os.path.join(MODEL_DIR, name)
    return joblib.load(path) if os.path.exists(path) else None

def get_eta_model():    return _load("eta_model.joblib")
def get_demand_model(): return _load("demand_model.joblib")
def get_demand_le():    return _load("demand_label_encoder.joblib")
def get_score_model():  return _load("score_model.joblib")
def get_score_scaler(): return _load("score_scaler.joblib")
'''
print(loader_code)
print("Copy this into backend/app/ml/loader.py")


import joblib, os, numpy as np

MODEL_DIR = os.getenv("MODEL_DIR", "/app/ml_models")

def _load(name):
    path = os.path.join(MODEL_DIR, name)
    return joblib.load(path) if os.path.exists(path) else None

def get_eta_model():    return _load("eta_model.joblib")
def get_demand_model(): return _load("demand_model.joblib")
def get_demand_le():    return _load("demand_label_encoder.joblib")
def get_score_model():  return _load("score_model.joblib")
def get_score_scaler(): return _load("score_scaler.joblib")

Copy this into backend/app/ml/loader.py


In [10]:
import os

print("=" * 55)
print("MA3 ML PIPELINE — COMPLETE")
print("=" * 55)
models = {
    "ETA (LSTM)":            ("eta_model.joblib",      "0.65 min MAE"),
    "Demand (XGBoost)":      ("demand_model.joblib",   "R² 0.9564"),
    "Driver (IsoForest)":    ("score_model.joblib",    "ROC-AUC 1.0"),
}
for name, (fname, metric) in models.items():
    size = os.path.getsize(f"{MODEL_DIR}/{fname}") / 1024
    print(f"  ✓ {name:<22} {metric:<15} {size:>8.1f} KB")

print("=" * 55)
print("\nNext steps:")
print("  1. Copy ml/models/ → backend/ml_models/")
print("  2. Update backend/app/ml/loader.py")
print("  3. docker compose up --build")
print("  4. Test: curl http://localhost:8000/predict/eta")
print("  5. Run:  python scripts/simulate_gps.py")
print("=" * 55)

MA3 ML PIPELINE — COMPLETE
  ✓ ETA (LSTM)             0.65 min MAE       437.2 KB
  ✓ Demand (XGBoost)       R² 0.9564          252.4 KB
  ✓ Driver (IsoForest)     ROC-AUC 1.0       2680.0 KB

Next steps:
  1. Copy ml/models/ → backend/ml_models/
  2. Update backend/app/ml/loader.py
  3. docker compose up --build
  4. Test: curl http://localhost:8000/predict/eta
  5. Run:  python scripts/simulate_gps.py
